# PT & OT Consult Orders
Creates CLIF *key_icu_orders* table from MIMIC data.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from datetime import datetime
from datetime import timedelta
import numpy as np
import os
import json

#Load file path
with open('../config/config.json', 'r') as file:
    config = json.load(file)
path = path = os.path.join(config['mimic'], 'icu', 'chartevents.parquet')
pyarr_file = pq.ParquetFile(path)

# Define chunk size (adjust based on available memory)
CHUNK_SIZE = 10000

mapping = {
    "PT": "pt_evaluation",
    "OT": "ot_evaluation"
}

processed_chunks = []

for batch in pyarr_file.iter_batches(batch_size=CHUNK_SIZE):
    chunk = batch.to_pandas()
    
    # Filter for PT orders only
    chart_mask = (
        chunk["hadm_id"].notna() &
        chunk["itemid"].notna() &
        (chunk["itemid"] == 225135) &
        chunk["value"].notna() &
        chunk["value"].isin(["PT", "OT"])
    )
    chunk = chunk[chart_mask]

    # Skip empty chunks
    if chunk.empty:
        continue

    # Convert string data types
    chunk['hospitalization_id'] = chunk['hadm_id'].astype(str)
    chunk['order_status'] = 'sent'

    # Convert times
    chunk['order_dttm'] = pd.to_datetime(chunk['charttime'], utc=False)
    chunk['order_dttm'] = chunk['order_dttm'].dt.tz_localize(
        'America/New_York', ambiguous=True, nonexistent='shift_forward'
    )
    chunk['order_dttm'] = chunk['order_dttm'].dt.tz_convert('UTC')

    # Convert order name and categorize
    chunk['order_name'] = chunk['value']
    chunk['order_category'] = chunk['order_name'].map(mapping)

    processed_chunks.append(chunk)

# Combine all chunks
pt_df = pd.concat(processed_chunks, ignore_index=True)

#Reorganize columns
pt_df = pt_df[['hospitalization_id','order_dttm','order_name','order_category','order_status']]
pt_df.dtypes

In [ ]:
#Save
import os
path = os.path.join(config['output_folder'], "clif_key_icu_orders.parquet")
pt_df.to_parquet(path)

In [ ]:
# Post-processing diagnostics
print(f"Shape: {pt_df.shape}")
print(f"Blank order times: {pt_df['order_dttm'].isna().sum()}")
print(pt_df['order_category'].value_counts())